In [101]:
import pandas as pd 
import seaborn as sns 
import numpy as np 
import matplotlib.pyplot as plt
import warnings 
warnings.filterwarnings('ignore')

In [102]:
df=pd.read_csv('train.txt',sep=';',header=None,names=['Text','emotions'])

In [103]:
df.head()

,Text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [104]:
df.isnull().sum()

Text        0
emotions    0
dtype: int64

In [105]:
df.duplicated().sum()

np.int64(1)

In [106]:
df.shape

(16000, 2)

In [107]:
df=df.drop_duplicates()

In [108]:
df.duplicated().sum()

np.int64(0)

In [109]:
df.shape

(15999, 2)

In [110]:
df['emotions'].value_counts()

emotions
joy         5361
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64

In [111]:
df['emotions'].unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [112]:
emo_map={
    "sadness":0,
    "anger":1,
    "love":2,
    "surprise":3,
    "fear":4,
    "joy":5
}
df['emotions']=df['emotions'].map(emo_map)

In [113]:
df['emotions'].value_counts()

emotions
5    5361
0    4666
1    2159
4    1937
2    1304
3     572
Name: count, dtype: int64

In [114]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15999 entries, 0 to 15999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Text      15999 non-null  object
 1   emotions  15999 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 375.0+ KB


In [115]:
print(df["Text"].str.islower().all())

True


In [116]:
df['Text']=df['Text'].str.lower()

In [117]:
import string 
def rem_punc(txt): 
    return txt.translate(str.maketrans('','',string.punctuation))
    

In [118]:
df['Text']=df['Text'].apply(rem_punc)

In [119]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['Text'] = df['Text'].apply(remove_emojis)

In [120]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [121]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\arpan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arpan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [122]:
stop_words = set(stopwords.words('english'))

In [123]:
print(stop_words)

{'don', 's', 'up', "you'd", 'further', 'below', 'ma', "hadn't", 'll', 'if', 'this', 'themselves', 'its', 'over', 'won', 'after', 'd', 'have', "they'd", 'which', 'against', 'y', "won't", "they've", 'had', "we'd", 'of', "he'll", 'all', 'being', 'does', "hasn't", "she'd", "haven't", 'before', 'between', "mightn't", 'as', 'itself', 'most', 'nor', 'why', 'here', "we're", 'whom', 'will', 'your', 'too', 'ours', 'about', 'should', 'for', 'them', 'than', 'down', "isn't", 'what', 'then', 'doesn', "weren't", 'weren', "needn't", 'can', 'was', 'shan', 'how', 'some', 'into', 'so', 'each', "it'd", 'few', 'doing', 'other', "she'll", 'be', "shan't", "she's", 'until', 'hadn', 'my', 'no', 'our', "wouldn't", "we'll", 'it', 'himself', "doesn't", 'just', 'haven', 'on', 'both', 'theirs', 'by', "he's", "i'm", 'mustn', "mustn't", 'an', 'her', 'he', 'above', 'again', "you've", 'o', 'in', 'do', 'did', "they'll", 'very', 'hers', 'where', 'didn', 'aren', "he'd", 'been', 'has', 'a', 'she', 'but', 'off', "it's", 'sh

In [127]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\arpan\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [124]:
df['Text'].loc[565]

'i have been feeling conflicted on whether or not i as a follower of christ should celebrate the ever popular pagan originated modern day holidays'

In [125]:
from nltk.tokenize import word_tokenize

def remove_stopwords(text):
    words = word_tokenize(text)

    cleaned = [
        word for word in words
        if word not in stop_words
    ]

    return " ".join(cleaned)

In [128]:
df["Text"] = df["Text"].apply(remove_stopwords)

In [129]:
df['Text'].loc[565]

'feeling conflicted whether follower christ celebrate ever popular pagan originated modern day holidays'

In [130]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [134]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['Text'], df['emotions'], test_size=0.20, random_state=42)

In [135]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.773125


In [136]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0], shape=(3200,))

In [137]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None
